In [3]:
from pathlib import Path
import json

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.general import GiveUniqueNodeNames


MODEL_PATH = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/"
    "finn_build_pynqz2(W4A4)/intermediate_models/step_specialize_layers.onnx"
)

OUT_JSON = Path(
    "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/"
    "manual_folding_config.json"
)

# Current MVAU order from your latest cycle report:
#
# idx  node        old cycles   suggested PE/SIMD
# 0    MVAU_hls_0  2240         PE=1, SIMD=1
# 1    MVAU_hls_1  6720         PE=1, SIMD=1
# 2    MVAU_hls_2  107520       PE=4, SIMD=1
# 3    MVAU_hls_3  107520       PE=4, SIMD=1
# 4    MVAU_hls_4  107520       PE=4, SIMD=1
# 5    MVAU_hls_5  107520       PE=4, SIMD=1
# 6    MVAU_hls_6  107520       PE=4, SIMD=1
# 7    MVAU_hls_7  2800         PE=1, SIMD=1
MANUAL_MVAU_CONFIGS = [
    {"PE": 1, "SIMD": 1},  # MVAU_hls_0, cycles=2240, 不动
    {"PE": 2, "SIMD": 1},  # MVAU_hls_1, cycles=6720, 不动

    {"PE": 16, "SIMD": 2},  # MVAU_hls_2, 107520 -> 26880
    {"PE": 16, "SIMD": 2},  # MVAU_hls_3, 107520 -> 26880
    {"PE": 16, "SIMD": 2},  # MVAU_hls_4, 107520 -> 26880
    {"PE": 16, "SIMD": 2},  # MVAU_hls_5, 107520 -> 26880
    {"PE": 16, "SIMD": 2},  # MVAU_hls_6, 107520 -> 26880

    {"PE": 1, "SIMD": 4},  # MVAU_hls_7, cycles=2800, 不动
]


model = ModelWrapper(str(MODEL_PATH))

# Important: your current node names are empty.
# This gives deterministic names such as MVAU_hls_0, MVAU_hls_1, ...
model = model.transform(GiveUniqueNodeNames())

folding_config = {
    "Defaults": {}
}
mvau_idx = 0

for node in model.graph.node:
    if node.op_type not in ["MVAU_hls", "MVAU_rtl"]:
        continue

    inst = getCustomOp(node)

    mw = int(inst.get_nodeattr("MW"))
    mh = int(inst.get_nodeattr("MH"))

    cfg = MANUAL_MVAU_CONFIGS[mvau_idx]
    pe = int(cfg["PE"])
    simd = int(cfg["SIMD"])

    assert mw % simd == 0, (
        f"{node.name}: invalid SIMD={simd} for MW={mw}. "
        f"Need MW % SIMD == 0."
    )
    assert mh % pe == 0, (
        f"{node.name}: invalid PE={pe} for MH={mh}. "
        f"Need MH % PE == 0."
    )

    folding_config[node.name] = {
        "PE": pe,
        "SIMD": simd,
        # keep weights in DSP for now
        "resType": "dsp",
        # leave memory style automatic
        "ram_style": "auto",
    }

    old_cycles = int(inst.get_exp_cycles())
    new_cycles_est = old_cycles / (pe * simd)

    print(f"[{mvau_idx}] {node.name}")
    print(f"  op_type: {node.op_type}")
    print(f"  MW/MH  : {mw}/{mh}")
    print(f"  PE/SIMD: {pe}/{simd}")
    print(f"  old cycles approx: {old_cycles}")
    print(f"  new cycles rough : {new_cycles_est:.1f}")
    print()

    mvau_idx += 1

if mvau_idx != len(MANUAL_MVAU_CONFIGS):
    raise RuntimeError(
        f"Found {mvau_idx} MVAU nodes, "
        f"but MANUAL_MVAU_CONFIGS has {len(MANUAL_MVAU_CONFIGS)} entries."
    )

with open(str(OUT_JSON), "w") as f:
    json.dump(folding_config, f, indent=2)

print("Saved manual folding config to:")
print(OUT_JSON)

[0] MVAU_hls_0
  op_type: MVAU_hls
  MW/MH  : 1/16
  PE/SIMD: 1/1
  old cycles approx: 2240
  new cycles rough : 2240.0

[1] MVAU_hls_1
  op_type: MVAU_hls
  MW/MH  : 3/16
  PE/SIMD: 2/1
  old cycles approx: 6720
  new cycles rough : 3360.0

[2] MVAU_hls_2
  op_type: MVAU_hls
  MW/MH  : 48/16
  PE/SIMD: 16/2
  old cycles approx: 107520
  new cycles rough : 3360.0

[3] MVAU_hls_3
  op_type: MVAU_hls
  MW/MH  : 48/16
  PE/SIMD: 16/2
  old cycles approx: 107520
  new cycles rough : 3360.0

[4] MVAU_hls_4
  op_type: MVAU_hls
  MW/MH  : 48/16
  PE/SIMD: 16/2
  old cycles approx: 107520
  new cycles rough : 3360.0

[5] MVAU_hls_5
  op_type: MVAU_hls
  MW/MH  : 48/16
  PE/SIMD: 16/2
  old cycles approx: 107520
  new cycles rough : 3360.0

[6] MVAU_hls_6
  op_type: MVAU_hls
  MW/MH  : 48/16
  PE/SIMD: 16/2
  old cycles approx: 107520
  new cycles rough : 3360.0

[7] MVAU_hls_7
  op_type: MVAU_hls
  MW/MH  : 2240/5
  PE/SIMD: 1/4
  old cycles approx: 2800
  new cycles rough : 700.0

Saved manua